In [1]:
import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import numpy as np

from src.environment import HotelPricingEnv

from src.config import TRAINING_EPISODES

from src.q_learning_utils import (initialize_q_table,state_to_index)

from src.q_learning import (choose_action,update_q_table)

In [3]:
env = HotelPricingEnv()
q_table = initialize_q_table()
print(q_table.shape)

(1581, 5)


In [4]:
episodes = TRAINING_EPISODES
for episode in range(episodes):
    state,_ = env.reset()
    done = False
    while not done:
        state_index = state_to_index(state[0],state[1])
        action = choose_action(q_table,state_index)
        next_state, reward, done, _, info = env.step(action)
        next_index = state_to_index(next_state[0],next_state[1])

        update_q_table(q_table,state_index,action,reward,next_index)
        
        state = next_state

In [5]:
updated = np.count_nonzero(q_table)

print("Updated Q-values :", updated)

print("Total Q-values   :", q_table.size)

print(f"Learning Progress : {(updated/q_table.size)*100:.2f}%")

Updated Q-values : 819
Total Q-values   : 7905
Learning Progress : 10.36%


In [6]:
sample_states = [
    (50, 30),
    (45, 25),
    (35, 20),
    (25, 15),
    (10, 5)
]

for rooms, days in sample_states:

    idx = state_to_index(rooms, days)

    print(f"\nState ({rooms}, {days})")

    print(q_table[idx])


State (50, 30)
[ 62.97224026  58.62400109  78.7425367   75.23916865 184.56848805]

State (45, 25)
[70.85274744  0.         33.56843523  1.045      13.5       ]

State (35, 20)
[10.74186227  0.          0.         64.97453727  0.        ]

State (25, 15)
[35.91256629 10.50705026 10.0225     24.15370612  0.        ]

State (10, 5)
[114.84095856  31.2733325    0.           0.           0.        ]


In [7]:
for rooms,days in sample_states:
    idx = state_to_index(rooms,days)
    best_action = np.argmax(q_table[idx])
    print(f"Rooms = {rooms}, Days = {days} -> Best Action = {best_action}")

Rooms = 50, Days = 30 -> Best Action = 4
Rooms = 45, Days = 25 -> Best Action = 0
Rooms = 35, Days = 20 -> Best Action = 3
Rooms = 25, Days = 15 -> Best Action = 0
Rooms = 10, Days = 5 -> Best Action = 0


In [8]:
visited_states = np.where(np.sum(q_table, axis=1) != 0)[0]

print(f"Visited States : {len(visited_states)}")
print(f"Total States   : {51 * 31}")
print(f"Coverage       : {(len(visited_states) / (51 * 31)) * 100:.2f}%")

Visited States : 528
Total States   : 1581
Coverage       : 33.40%
